# Google Colab - small models on all 5 benchmarks

Runs the **small local-PC team** `Qwen/Qwen2.5-0.5B-Instruct` + `Qwen/Qwen2.5-1.5B-Instruct` (cycled as `0.5, 1.5, 0.5`) on:

- PubMedQA (`pqa_labeled`, genuine H4 evidence partition)
- MedQA
- MMLU-Med
- MedMCQA (validation)
- MedXpertQA (text)

300 cases x `H0,H1,H2,H3,H4,H5,SYC`. Each dataset has its own config and output dir, so `--resume` is safe.

**Before running:** push the latest repo from your laptop to GitHub.  
**If Colab disconnects:** re-run the setup cells and the full-sweep cell below; it continues from the last saved record in Google Drive.  
**Runtime:** Runtime -> Change runtime type -> **T4 GPU**. No API key needed.

In [ ]:
import os
REPO = "/content/Multi-Agent-medical-diagnosis-llm"
if not os.path.exists(os.path.join(REPO, ".git")):
    !git clone -q https://github.com/HanjalaShihab/Multi-Agent-medical-diagnosis-llm.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}/pilot
print(os.getcwd())

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU -> Runtime > Change runtime type > T4 GPU")

In [ ]:
!pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

In [ ]:
!python -m bts_audit.cli doctor --config configs/colab_pubmedqa_small.yaml

In [ ]:
!mkdir -p data/cases
!python scripts/download_data.py all --out data/raw
!python -m bts_audit.cli prepare --dataset pubmedqa   --raw data/raw/pubmedqa   --out data/cases/pubmedqa_300.jsonl   --n 300
!python -m bts_audit.cli prepare --dataset medqa      --raw data/raw/medqa      --out data/cases/medqa_300.jsonl      --n 300
!python -m bts_audit.cli prepare --dataset mmlu_med   --raw data/raw/mmlu       --out data/cases/mmlu_med_300.jsonl   --n 300
!python -m bts_audit.cli prepare --dataset medmcqa    --raw data/raw/medmcqa    --out data/cases/medmcqa_300.jsonl    --n 300
!python -m bts_audit.cli prepare --dataset medxpertqa --raw data/raw/medxpertqa --out data/cases/medxpertqa_300.jsonl --n 300

## Persist results in Google Drive

Mount Drive so `answers.jsonl` / `forecasts.jsonl` survive Colab disconnects and session resets. Re-run this cell after a reset, then run the sweep cell with the same `OUT_ROOT`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_ROOT = "/content/drive/MyDrive/bts_audit_colab_results"
!mkdir -p '{OUT_ROOT}'
print(OUT_ROOT)

## Optional quick first pass (60 cases, `H1,H4,H5,SYC`)
Skip this cell if you want to go straight to the full sweep.

In [ ]:
!OUT_ROOT='/content/drive/MyDrive/bts_audit_colab_results' LIMIT=60 CELLS=H1,H4,H5,SYC RESUME=1 bash scripts/run_colab_all_small.sh

## Full sweep - all datasets, 300 cases, `H0,H1,H2,H3,H4,H5,SYC`

If the session disconnects or the VM restarts, re-run the setup cells and **this same cell**. `--resume` continues from the last record saved in Google Drive.

In [ ]:
!OUT_ROOT='/content/drive/MyDrive/bts_audit_colab_results' LIMIT=300 CELLS=H0,H1,H2,H3,H4,H5,SYC RESUME=1 bash scripts/run_colab_all_small.sh

In [ ]:
from pathlib import Path
import pandas as pd
root = Path("/content/drive/MyDrive/bts_audit_colab_results")
for d in sorted(root.glob("colab_*_small")):
    print("\n===", d, "===")
    cm = d / "cell_metrics.csv"
    if not cm.exists():
        print("no cell_metrics.csv")
        continue
    df = pd.read_csv(cm)
    cols = [c for c in ["cell","n_cases","accuracy","selection_accuracy_majority","selection_accuracy_confidence","selection_accuracy_bts","selection_accuracy_proper","auc_score","auc_confidence","delta_auc","agreement_rate","rho","delta_auc_oracle"] if c in df.columns]
    print(df[cols].to_string(index=False))

In [ ]:
!zip -rq /content/colab_small_results.zip '/content/drive/MyDrive/bts_audit_colab_results'
from google.colab import files
files.download("/content/colab_small_results.zip")